# GüvenLoc — Faz 1, Kaggle (T4 ×2)

Ayrıntılı açıklama: `docs/faz1_kaggle.md`. Bu defter iki modda çalışır:

| MOD | Ne yapar | Süre |
|---|---|---|
| `"olcum"` | Kurulum + testler + T4 hız ölçümü + MegaDepth duman testi (25 çift, orijinalle birebir karşılaştırma) | ≈ 30–60 dk |
| `"repro"` | MegaDepth-1500 (GPU0) ve C4 (GPU1) tam tekrarlama, tohum tohum; 11. saatte kendini güvenle durdurur | ≤ 11 sa |

**Ayarlar (sağ panel):** Accelerator = **GPU T4 x2**, Internet = **On**.
Önceki oturumun sonuçlarını taşımak için: *Add Input → Your Work → bu defterin önceki sürümü*.

In [ ]:
MODE = "olcum"            # "olcum" (ilk oturum) | "repro" (uzun koşu)
SEEDS = [0, 1, 2, 3, 4]   # biten tohumlar otomatik atlanır
BRANCH = "faz1/altyapi"
BUDGET_H = 11.0           # oturum sınırı 12 sa; bundan sonra koşular durdurulur (biten tohumlar korunur)

import glob
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

T_START = time.time()
WORK = "/kaggle/working"
REPO = f"{WORK}/guvenloc"
MD = "/tmp/megadepth"

def sh(cmd, cwd=REPO, check=True):
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, cwd=cwd, text=True, check=False)
    if check and r.returncode != 0:
        raise RuntimeError(f"komut başarısız ({r.returncode}): {cmd}")
    return r.returncode

## 1. Kurulum

In [ ]:
if not os.path.isdir(REPO):
    sh(f"git clone -q https://github.com/efsunboz16/guvenloc.git {REPO}", cwd=WORK)
sh(f"git fetch -q && git checkout -q {BRANCH} && git pull -q && git log --oneline -1")
# cu128 Kaggle sürücüsünde açılmazsa aşağıda cu128 -> cu126 yapın
sh("pip install -q torch==2.11.0 torchvision==0.26.0 --index-url https://download.pytorch.org/whl/cu128")
sh("pip install -q -r requirements.txt")
sh(f"{sys.executable} scripts/00_check_env.py")

## 2. MegaDepth-1500 test verisi (≈ birkaç GB, her oturumda /tmp'ye)

In [ ]:
BASE = "https://github.com/Parskatt/storage/releases/download/mega1500"
NPZ = ["0015_0.1_0.3.npz", "0015_0.3_0.5.npz", "0022_0.1_0.3.npz", "0022_0.3_0.5.npz", "0022_0.5_0.7.npz"]
os.makedirs(MD, exist_ok=True)
for f in NPZ:
    if not os.path.exists(f"{MD}/{f}"):
        sh(f"wget -q {BASE}/{f} -O {MD}/{f}", cwd=MD)
if not os.path.isdir(f"{MD}/Undistorted_SfM"):
    sh(f"wget -q {BASE}/megadepth_test_1500.tar -O /tmp/md.tar", cwd="/tmp")
    sh("tar -xf /tmp/md.tar -C /tmp && mv /tmp/megadepth_test_1500/Undistorted_SfM " + MD
       + " && rm -rf /tmp/md.tar /tmp/megadepth_test_1500", cwd="/tmp")
print("MegaDepth hazır:", sorted(os.listdir(MD)))

## 3. Önceki oturumların sonuçlarını geri taşı (varsa)

In [ ]:
os.makedirs(f"{REPO}/results/faz1", exist_ok=True)
n = 0
for f in glob.glob("/kaggle/input/**/results/faz1/repro_*_seed*.json", recursive=True):
    dst = f"{REPO}/results/faz1/{os.path.basename(f)}"
    if not os.path.exists(dst):
        shutil.copy(f, dst); n += 1
prev = sorted(glob.glob("/kaggle/input/**/results/runs.csv", recursive=True))
if prev:
    shutil.copy(prev[0], f"{REPO}/results/runs.csv")
print(f"{n} tohum dosyası taşındı; runs.csv: {prev[:1]}")
print(sorted(os.path.basename(f) for f in glob.glob(f"{REPO}/results/faz1/repro_*_seed*.json")))

## 4. MOD = "olcum": testler, hız, duman testi
UAV-VisLoc gerekmez. Yalnızca önbellek hızını ölçmek isterseniz bölge **04** klasörü +
`satellite_ coordinates_range.csv`'yi küçük bir Dataset olarak (≈ 2,6 GB) ekleyin; yoksa hız ölçümü atlanır.

In [ ]:
if MODE == "olcum":
    sh(f"{sys.executable} -m pytest -q", check=False)
    sh(f"CUDA_VISIBLE_DEVICES=0 {sys.executable} scripts/20_repro_mega1500.py --data {MD} --limit 5 --check-original")
    r = json.loads(Path(f"{REPO}/results/faz1/repro_mega1500_lim5_seed0.json").read_text())
    seed_h = r["sec"] / r["n_pairs"] * 1500 / 3600
    print(f"\n>>> T4'te bir tam tohum ≈ {seed_h:.1f} sa; bir oturumda GPU başına ≈ {int(BUDGET_H // seed_h)} tohum")
    rng = glob.glob("/kaggle/input/**/satellite*coordinates*range*.csv", recursive=True)
    if rng:
        sh(f"{sys.executable} scripts/23_speed.py '{os.path.dirname(rng[0])}' --settings base mega1500 --region 04")
        sp = {x["setting"]: x for x in json.loads(Path(f"{REPO}/results/faz1/speed.json").read_text())["results"]}
        for s, x in sp.items():
            print(f">>> önbellek {s}: {x['matcher_s']} s/dosya -> tam önbellek iki T4'te ≈ "
                  f"{86000 * x['matcher_s'] / 2 / 3600:.0f} sa")
    else:
        print("UAV-VisLoc eklenmemiş: önbellek hız ölçümü atlandı")

## 5. MOD = "repro": MegaDepth-1500 (GPU0) + C4 (GPU1), paralel

In [ ]:
def run_parallel(jobs):
    os.makedirs(f"{REPO}/logs", exist_ok=True)
    procs = []
    for gpu, args, log in jobs:
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED="1")
        procs.append((subprocess.Popen(f"{sys.executable} scripts/20_repro_mega1500.py {args}",
                                       shell=True, cwd=REPO, env=env,
                                       stdout=open(f"{REPO}/logs/{log}", "w"),  # noqa: SIM115
                                       stderr=subprocess.STDOUT), log))
    while any(p.poll() is None for p, _ in procs):
        time.sleep(600)
        el = (time.time() - T_START) / 3600
        for p, log in procs:
            last = Path(f"{REPO}/logs/{log}").read_text().strip().splitlines()[-1:]
            print(f"[{el:.1f} sa] {log}: {last}", flush=True)
        if el > BUDGET_H:
            print("Bütçe doldu: koşular durduruluyor (biten tohumlar kaydedildi)")
            for p, _ in procs:
                p.kill()
    for p, log in procs:
        print(log, "çıkış kodu", p.returncode)

if MODE == "repro":
    s = " ".join(map(str, SEEDS))
    run_parallel([(0, f"--data {MD} --seeds {s}", "mega1500_gpu0.log"),
                  (1, f"--data {MD} --c4 --seeds {s}", "c4_gpu1.log")])

## 6. Özet ve indirilecek dosya

In [ ]:
for name, flag in (("mega1500", ""), ("c4", "--c4")):
    done = sorted(int(f.split("seed")[-1][:-5])
                  for f in glob.glob(f"{REPO}/results/faz1/repro_{name}_seed*.json"))
    print(name, "biten tohumlar:", done)
    if done and MODE == "repro":
        sh(f"{sys.executable} scripts/20_repro_mega1500.py --data {MD} {flag} --seeds {' '.join(map(str, done))}",
           check=False)   # hepsi atlanır, yalnızca özet + runs.csv üretilir
sh(f"cd {REPO} && zip -qr {WORK}/sonuclar.zip results logs 2>/dev/null; ls -la {WORK}/sonuclar.zip", check=False)
print("Sağ panel -> Output -> sonuclar.zip indirin.")